# Phase 1 — Schema Definition & Landing/Raw Layers

**Scope of this notebook:** Yellow Taxi, January 2024 only.

Goals:
1. Load the landing-layer parquet file with PySpark (landing files are never modified).
2. Inspect the inferred schema and compare it against a canonical schema derived from the TLC Yellow Taxi data dictionary (`resources/z NYC Yellow Taxi Trip Records Data Dictionary.pdf`).
3. Check missingness per column.
4. Identify outliers / business-rule violations per column (counts only — **no filtering or cleaning happens here**, that belongs to the Raw → Clean pipeline, `src/cleaning.py`).

This notebook is exploratory. Once the schema and outlier patterns are understood across more months, the reusable ingestion/standardisation logic should move into `src/dataset.py`.

In [1]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (
    SparkSession.builder
    .appName("nyc-taxi-schema-raw-layer")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/28 15:58:40 WARN Utils: Your hostname, Debaos-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.2 instead (on interface en0)
26/08/28 15:58:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/28 15:58:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


## 1. Load landing data (Yellow Taxi, Jan 2024)

In [2]:
yellow_jan_2024_path = "../data/landing/yellow/Yellow Taxi Data Jan 2024.parquet"

yellow_jan_2024 = spark.read.parquet(yellow_jan_2024_path)

row_count = yellow_jan_2024.count()
col_count = len(yellow_jan_2024.columns)
print(f"Rows: {row_count:,}")
print(f"Columns: {col_count}")

Rows: 2,964,624
Columns: 19


## 2. Inferred schema

In [3]:
yellow_jan_2024.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



## 3. Canonical schema (from TLC data dictionary)

Defined by hand from `resources/z NYC Yellow Taxi Trip Records Data Dictionary.pdf`. This is used only as a *reference for comparison* in this notebook — we are not yet forcing this schema on read (that decision is deferred until multiple months have been inspected).

In [4]:
canonical_yellow_schema = T.StructType([
    T.StructField("VendorID", T.IntegerType(), True),
    T.StructField("tpep_pickup_datetime", T.TimestampType(), True),
    T.StructField("tpep_dropoff_datetime", T.TimestampType(), True),
    T.StructField("passenger_count", T.LongType(), True),
    T.StructField("trip_distance", T.DoubleType(), True),
    T.StructField("RatecodeID", T.LongType(), True),
    T.StructField("store_and_fwd_flag", T.StringType(), True),
    T.StructField("PULocationID", T.IntegerType(), True),
    T.StructField("DOLocationID", T.IntegerType(), True),
    T.StructField("payment_type", T.LongType(), True),
    T.StructField("fare_amount", T.DoubleType(), True),
    T.StructField("extra", T.DoubleType(), True),
    T.StructField("mta_tax", T.DoubleType(), True),
    T.StructField("tip_amount", T.DoubleType(), True),
    T.StructField("tolls_amount", T.DoubleType(), True),
    T.StructField("improvement_surcharge", T.DoubleType(), True),
    T.StructField("total_amount", T.DoubleType(), True),
    T.StructField("congestion_surcharge", T.DoubleType(), True),
    T.StructField("Airport_fee", T.DoubleType(), True),
])

inferred_fields = {f.name: f.dataType.simpleString() for f in yellow_jan_2024.schema.fields}
canonical_fields = {f.name: f.dataType.simpleString() for f in canonical_yellow_schema.fields}

print("Columns in landing file but not in canonical schema:")
print(sorted(set(inferred_fields) - set(canonical_fields)) or "(none)")

print("\nColumns in canonical schema but missing from landing file:")
print(sorted(set(canonical_fields) - set(inferred_fields)) or "(none)")

print("\nType mismatches (column: inferred vs canonical):")
for name in sorted(set(inferred_fields) & set(canonical_fields)):
    if inferred_fields[name] != canonical_fields[name]:
        print(f"  {name}: {inferred_fields[name]} vs {canonical_fields[name]}")

Columns in landing file but not in canonical schema:
(none)

Columns in canonical schema but missing from landing file:
(none)

Type mismatches (column: inferred vs canonical):
  tpep_dropoff_datetime: timestamp_ntz vs timestamp
  tpep_pickup_datetime: timestamp_ntz vs timestamp


## 4. Missingness per column

In [5]:
null_counts = yellow_jan_2024.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in yellow_jan_2024.columns
]).collect()[0].asDict()

print(f"{'column':<24}{'nulls':>12}{'pct_null':>12}")
for col_name, n_null in null_counts.items():
    pct = 100 * n_null / row_count
    print(f"{col_name:<24}{n_null:>12,}{pct:>11.2f}%")

column                         nulls    pct_null
VendorID                           0       0.00%
tpep_pickup_datetime               0       0.00%
tpep_dropoff_datetime              0       0.00%
passenger_count              140,162       4.73%
trip_distance                      0       0.00%
RatecodeID                   140,162       4.73%
store_and_fwd_flag           140,162       4.73%
PULocationID                       0       0.00%
DOLocationID                       0       0.00%
payment_type                       0       0.00%
fare_amount                        0       0.00%
extra                              0       0.00%
mta_tax                            0       0.00%
tip_amount                         0       0.00%
tolls_amount                       0       0.00%
improvement_surcharge              0       0.00%
total_amount                       0       0.00%
congestion_surcharge         140,162       4.73%
Airport_fee                  140,162       4.73%


## 5. Numeric summary statistics

`min` / `max` here are the first pass at spotting outliers (e.g. negative fares, zero/huge trip distances).

In [6]:
numeric_cols = [
    "passenger_count", "trip_distance", "fare_amount", "extra", "mta_tax",
    "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount",
    "congestion_surcharge", "Airport_fee",
]

yellow_jan_2024.select(numeric_cols).summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
).show(truncate=False)

+-------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+
|summary|passenger_count   |trip_distance     |fare_amount       |extra             |mta_tax            |tip_amount        |tolls_amount      |improvement_surcharge|total_amount      |congestion_surcharge|Airport_fee        |
+-------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+
|count  |2824462           |2964624           |2964624           |2964624           |2964624            |2964624           |2964624           |2964624              |2964624           |2824462             |2824462            |
|mean   |1.3392808966805005|3.6521691789583146|18.175061916791037|1.4515984320439959|0.483382310

## 6. Categorical / coded field distributions

Codes outside the ranges defined in the data dictionary are outliers for these fields.

In [7]:
for col_name in ["VendorID", "RatecodeID", "payment_type", "store_and_fwd_flag"]:
    print(f"--- {col_name} ---")
    yellow_jan_2024.groupBy(col_name).count().orderBy(col_name).show(20, truncate=False)

--- VendorID ---


+--------+-------+
|VendorID|count  |
+--------+-------+
|1       |729732 |
|2       |2234632|
|6       |260    |
+--------+-------+

--- RatecodeID ---
+----------+-------+
|RatecodeID|count  |
+----------+-------+
|NULL      |140162 |
|1         |2663350|
|2         |98713  |
|3         |7954   |
|4         |6365   |
|5         |19410  |
|6         |7      |
|99        |28663  |
+----------+-------+

--- payment_type ---


+------------+-------+
|payment_type|count  |
+------------+-------+
|0           |140162 |
|1           |2319046|
|2           |439191 |
|3           |19597  |
|4           |46628  |
+------------+-------+

--- store_and_fwd_flag ---


+------------------+-------+
|store_and_fwd_flag|count  |
+------------------+-------+
|NULL              |140162 |
|N                 |2813126|
|Y                 |11336  |
+------------------+-------+



## 7. Business-rule outlier checks

Reference ranges, from the data dictionary and TLC documentation:
- `VendorID` valid codes: {1, 2, 6, 7}
- `RatecodeID` valid codes: {1, 2, 3, 4, 5, 6, 99}
- `payment_type` valid codes: {0, 1, 2, 3, 4, 5, 6}
- `store_and_fwd_flag` valid values: {Y, N}
- `PULocationID` / `DOLocationID` valid TLC taxi zones: 1–263, with 264/265 reserved for "Unknown"/"N/A"
- Monetary fields (`fare_amount`, `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, `improvement_surcharge`, `total_amount`, `congestion_surcharge`, `Airport_fee`) should not be negative
- `trip_distance` should be positive
- `passenger_count` should be positive (0 passengers is not a valid trip)
- `tpep_dropoff_datetime` should not be before `tpep_pickup_datetime`
- Trip duration should be plausible (not zero-length, not multi-day)
- Pickup timestamps should fall within January 2024 (landing files can occasionally contain stray records from adjacent months)

In [8]:
trip_duration_min = (
    F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")
) / 60.0

checks = {
    "VendorID not in {1,2,6,7}": ~F.col("VendorID").isin(1, 2, 6, 7),
    "RatecodeID not in {1,2,3,4,5,6,99}": ~F.col("RatecodeID").isin(1, 2, 3, 4, 5, 6, 99),
    "payment_type not in {0,1,2,3,4,5,6}": ~F.col("payment_type").isin(0, 1, 2, 3, 4, 5, 6),
    "store_and_fwd_flag not in {Y,N}": ~F.col("store_and_fwd_flag").isin("Y", "N"),
    "passenger_count <= 0": F.col("passenger_count") <= 0,
    "passenger_count > 6": F.col("passenger_count") > 6,
    "trip_distance <= 0": F.col("trip_distance") <= 0,
    "trip_distance > 100 miles": F.col("trip_distance") > 100,
    "fare_amount < 0": F.col("fare_amount") < 0,
    "extra < 0": F.col("extra") < 0,
    "mta_tax < 0": F.col("mta_tax") < 0,
    "tip_amount < 0": F.col("tip_amount") < 0,
    "tolls_amount < 0": F.col("tolls_amount") < 0,
    "improvement_surcharge < 0": F.col("improvement_surcharge") < 0,
    "total_amount < 0": F.col("total_amount") < 0,
    "congestion_surcharge < 0": F.col("congestion_surcharge") < 0,
    "Airport_fee < 0": F.col("Airport_fee") < 0,
    "PULocationID out of [1,265]": ~F.col("PULocationID").between(1, 265),
    "DOLocationID out of [1,265]": ~F.col("DOLocationID").between(1, 265),
    "dropoff before pickup": F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime"),
    "trip duration == 0 min": trip_duration_min == 0,
    "trip duration > 1440 min (24h)": trip_duration_min > 1440,
    "pickup outside Jan 2024": (
        (F.year("tpep_pickup_datetime") != 2024) | (F.month("tpep_pickup_datetime") != 1)
    ),
}

agg_exprs = [
    F.sum(condition.cast("int")).alias(name.replace(" ", "_"))
    for name, condition in checks.items()
]

result_row = yellow_jan_2024.agg(*agg_exprs).collect()[0].asDict()

print(f"{'check':<40}{'count':>12}{'pct_of_rows':>14}")
for (name, _), count in zip(checks.items(), result_row.values()):
    count = count or 0
    pct = 100 * count / row_count
    print(f"{name:<40}{count:>12,}{pct:>13.3f}%")

check                                          count   pct_of_rows
VendorID not in {1,2,6,7}                          0        0.000%
RatecodeID not in {1,2,3,4,5,6,99}                 0        0.000%
payment_type not in {0,1,2,3,4,5,6}                0        0.000%
store_and_fwd_flag not in {Y,N}                    0        0.000%
passenger_count <= 0                          31,465        1.061%
passenger_count > 6                               60        0.002%
trip_distance <= 0                            60,371        2.036%
trip_distance > 100 miles                         59        0.002%
fare_amount < 0                               37,448        1.263%
extra < 0                                     17,548        0.592%
mta_tax < 0                                   34,434        1.161%
tip_amount < 0                                   102        0.003%
tolls_amount < 0                               2,035        0.069%
improvement_surcharge < 0                     35,502        1.

## 8. Notes / findings

**Schema:** The inferred schema matches the canonical (data-dictionary) schema on every column name. The only mismatch is `tpep_pickup_datetime` / `tpep_dropoff_datetime` being `timestamp_ntz` rather than `timestamp` — this is a Spark 4 parquet-inference quirk, not a real schema problem, but it's worth remembering if it causes cast issues later (e.g. `unix_timestamp()` works, `.cast("long")` does not on `timestamp_ntz`).

**Missingness:** `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge`, and `Airport_fee` are all null together on the same 140,162 rows (4.73%) — this looks like one batch of records missing a consistent set of fields, not independent random missingness. Worth checking if these correspond to a specific vendor or a specific time window.

**Outliers worth a cleaning-pipeline decision:**
- `trip_distance <= 0` (2.04%) and `trip_distance > 100 miles` (max recorded: 312,722 miles — clearly erroneous) — need a defensible upper/lower bound.
- `passenger_count <= 0` (1.06%) — 0-passenger trips are not valid; also `passenger_count == 9` occurs (max), which is plausible for vans but rare.
- Negative monetary fields (`fare_amount`, `mta_tax`, `improvement_surcharge`, `total_amount`, `extra`, `tolls_amount`, `tip_amount`, `congestion_surcharge`, `Airport_fee`) all occur at ~0.6–1.3% each — likely refunds/corrections in the source data, common across TLC trip data and usually dropped or taken as absolute value depending on convention.
- `dropoff before pickup` (56 rows) and `trip duration == 0 min` (814 rows) are physically invalid and should be excluded.
- `trip duration > 24h` (16 rows) is implausible for a taxi trip.
- `pickup outside Jan 2024` (18 rows) — stray records from an adjacent month leaking into this landing file; confirms the file isn't perfectly partitioned by month.
- All coded fields (`VendorID`, `RatecodeID`, `payment_type`, `store_and_fwd_flag`, `PULocationID`, `DOLocationID`) had **zero** out-of-range values in this file — good sign for schema reliability, but still needs checking on other months before assuming it always holds.

**Reminder:** this notebook only *identifies* issues. No rows are filtered or modified here — the landing file is read-only, and cleaning decisions belong to the Raw → Clean pipeline (`src/cleaning.py`, `src/clean_rules.py` — see notebook 02, `02_raw_to_clean.ipynb`), applied consistently across all months rather than ad hoc per file.

**Suggested next step:** repeat this same schema/outlier inspection on a few more Yellow Taxi months (e.g. a mid-2024 and a mid-2025 file) and on a Green Taxi file, to see whether the schema and missingness pattern above are stable or change over time — don't build the canonical ingestion function until that's confirmed.